In [1]:
import os
import openai
import json
import matplotlib.pyplot as plt
import ast
import h5py
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
def get_steps(user_query):
    system_prompt = """You are a medical image analysis expert. Your task is to break down a medical image analysis request into sequential processing steps.
    
    VALID TISSUE TYPES (examples):
    [
    "tumor",
    "necrotic_tissue",
    "normal_tissue",
    "epithelial_tissue",
    "connective_tissue",
    "muscle_tissue",
    "nervous_tissue",
    "vascular_tissue",
    "adipose_tissue",
    "lymphatic_tissue",
    "bone_tissue",
    "cartilage_tissue",
    "fibrous_tissue",
    "glandular_tissue",
    "scar_tissue",
    "granulation_tissue",
    "mesenchymal_tissue",
    "endothelial_tissue"
    ]

    VALID CELL TYPES (examples):
    [
    "podocyte",
    "hepatocyte",
    "fibroblast",
    "endothelial_cell",
    "epithelial_cell",
    "keratinocyte",
    "melanocyte",
    "chondrocyte",
    "osteoblast",
    "osteoclast",
    "osteocyte",
    "myocyte",
    "cardiomyocyte",
    "smooth_muscle_cell",
    "skeletal_muscle_fiber",
    "neuron",
    "astrocyte",
    "oligodendrocyte",
    "microglia",
    "Schwann_cell",
    "macrophage",
    "monocyte",
    "neutrophil",
    "basophil",
    "eosinophil",
    "lymphocyte",
    "B_cell",
    "T_cell",
    "NK_cell",
    "erythrocyte",
    "megakaryocyte",
    "platelet",
    "beta_cell_of_pancreas",
    "alpha_cell_of_pancreas",
    "kupffer_cell"
    ]   
    OUTPUT REQUIREMENTS:
    - Respond a string of steps that can be parsed by python's ast.literal_eval()
    - Each step must follow the exact format: "step N": {"model": "<model_type>", "input": "<target_name>"}
    - model_type must be either "TissueSeg", "NucleiSeg", or "Scripts" (no other values allowed)
    - Steps must be numbered sequentially starting from 1
    - input names MUST ONLY use the valid tissue types for TissueSeg and valid cell types for NucleiSeg as listed above
    - Also consider related tissues that would be necessary for comprehensive analysis
    - have high level vision of the analysis, like for cancer or metastasis queries, you may need to segment the tumor even if the user does not ask for it.
    
    STEP ORDERING RULES:
        1. Tissue segmentation must always precede nuclei detection for the same region
        2. Parent structures should be segmented before their sub-components
        3. For multiple tissues without dependencies, maintain a consistent order
        4. Scripts operations must come after all required segmentations are completed

    DECISION CRITERIA:
        1. Tissue Segmentation (TissueSeg) is needed when:
        - Regions of interest need to be isolated
        - Quantitative analysis of tissue areas is required
        - input must be a tissue name (at least as specific as like tumor, kidney, etc., something like metastasis area is not allowed)

        2. Nuclei Segmentation (NucleiSeg) is needed ONLY when:
        - Cell counting is requested clearly by the user
        - input must be a cell type name (something like metastasis is not allowed)

        3. Scripts is needed when:
        - Spatial relationships between segmented regions need to be analyzed
        - Calculations or operations need to be performed on segmented regions
        - Input should be a clear description of the operation to perform

    Here are some tips about solutions:

        1. Detection: Identify [certain cell type]

            Examples:

            * Cervix: Please find out all plasma cells on this image.
                * Solution: [Cell segmentation] → [Cell classification: Plasma cell] → [Calculation] → [Visualization]

        2. Detection: Identify [certain tissue type]

            Examples:

            * Breast: Please segment the invasive tumor in this breast biopsy and measure its area.
                * Solution: [Tissue Segmentation: Invasive tumor] → [Calculation] → [Visualization]
            * Prostate: Identify and label all glands in this prostate tissue section, then measure average glandular diameter.
                * Solution: [Tissue Segmentation: gland] → [Calculation] → [Visualization]

        3. Relational Detection: Identify the [certain cell type] on [certain tissue type]

            Examples:

            * Kidney: Can you help me count the average number of podocytes on kidney glomorulus?
                * Solution: [Tissue Segmentation: Glomorulus] → [Cell Segmentation] → [Cell Classification: Podocytes] → [Calculation] → [Visualization]
            * Colon: Find out the glands that contains mitosis.
                * Solution: [Tissue Segmentation: Gland] → [Cell Segmentation] → [Cell Classification: Mitosis] → [Calculation] → [Visualization]
            * Bone marrow: Can you find and calculate the plasma cell percentage in these bone marrow core biopsy slides?
                * Solution: [Tissue Segmentation: bone marrow] → [Cell Segmentation] → [Cell Classification: Plasma cell] → [Calculation] → [Visualization]

        4. Relational Detection: Identify the [certain tissue type] on [certain tissue type]

            Examples:

            * Lymph node: Can you help me find how many of the lymph nodes in this slide has tumor metastasis?
                * Solution: [Tissue Segmentation: Lymph node] → [Tissue Segmentation: Tumor] → [Calculation] → [Visualization]
            * Skin: What is the deepest depth of invasion for the melanoma in this skin biopsy?
                * Solution: [Tissue Segmentation: Melanoma] → [Tissue Segmentation: Skin epithelium surface] → [Calculation] → [Visualization]
            * Generic: Measure the distance from the tumor boundary to the nearest lymphatic vessel.
                * Solution: [Tissue Segmentation: Tumor] → [Tissue Segmentation: Lymphatic vessel] → [Calculation] → [Visualization]
            * Generic: Quantify the density of TILs at the tumor margin and within the tumor core for immunotherapy assessment.


    Example valid response:
        {
            'step 1': {'model': 'TissueSeg', 'input': 'lymph node'},
            'step 2': {'model': 'TissueSeg', 'input': 'tumor'},
            'step 3': {'model': 'Scripts', 'input': 'For every contour of lymph node, check which contour contains contours of tumor.'}
        }"""

    response = client.chat.completions.create(model="gpt-4o",
                                              messages=[{
                                                  "role":
                                                  "system",
                                                  "content":
                                                  system_prompt
                                              }, {
                                                  "role": "user",
                                                  "content": user_query
                                              }])
    print(response)
    return response.choices[0].message.content

In [3]:
user_query = "Can you help me count the average number of podocytes in kidney glomerulus?"
result = get_steps(user_query)
print(ast.literal_eval(result))

In [4]:
script_agent = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [13]:
def test_script(user_query):
    system_prompt = """
        You are an AI expert specializing in medical image analysis and geometric processing. Your task is to write Python code that processes segmentation results and performs specific analytical tasks on medical imaging data.
        You recieve a string that contains the task description.
        You is required to give a python code. The code will finish the task by processing a h5 file (as the function's argument) and return a dictionary of results.
        You ONLY need to return the code, do not include any other text even the python code block like ```python. Also, do notexecute the function (but you can give example input).
        Make the function name as "analyze_medical_image".
        
        Please follow these steps:
        1. Carefully read the problem statement, specifications, h5 file structure, and tips below.
        2. Explain your approach to solving the analytical task.
        3. Write a function that implements the analysis according to your plan, and return a dictionary of results.
        4. Use python default libraries and tools. You can also use h5py, numpy, torch, and some other common libraries.
        5. The function should take the path of the h5 file as the single argument.

        H5 FILE STRUCTURE:
        Format of H5 file
            Root
            |__ nuclei_segmentation
                |__ contours (N*32*2 - 3D array)
                |__ centroids (N*2 - 2D array)
                |__ class_vector (N*1 - 1D array)
                |__ class_names (a JSON string like this: {"0": "negative_control", "1": "tumor"})
            |__ tissue_segmentation
                |__ object_dictionary (see below)
            |__ patch_results
                |__ object_dictionary (see below)
            |__ wsi_results
                |__ object_dictionary (see below)

            Tissue_segmentation object dictionary (example):
            {“0”:{
                    “type”: “polygon”,
                    “coordinates_x”: [x1, x2, x3, x4, …, x10],
                    “coordinates_y”: [y1, y2, …, y10]
                },
            “1”: …
            }

            Patch results object dictionary (example):
            {“0”:{
                    “bbox”: [x1, y1, x2, y2],
                    “cosine_similarity": {
                        "tumor": 0.75,
                        "stromal": 0.12,
                        "lymphocytes": 0.3,
                    },
                    “probability": {
                        "tumor": 0.75,
                        "stromal": 0.12,
                        "lymphocytes": 0.3,
                    },
                    "final_class": "tumor",
                    "embedding": NP array
                },
            “1”:{
                    “bbox”: [x1, y1, x2, y2],
                    “cosine_similarity": {
                        "tumor": 0.25,
                        "stromal": 0.12,
                        "lymphocytes": 0.21,
                    },
                    “probability": {
                        "tumor": 0.25,
                        "stromal": 0.12,
                        "lymphocytes": 0.21,
                    },
                    "final_class": "tumor",
                    "embedding": NP array
                },
            "2": ...
            }

            WSI results object dictionary (example):
            {“0”:{
                    “cosine_similarity": {
                        "tumor": 0.75,
                        "stromal": 0.12,
                        "lymphocytes": 0.3,
                    },
                    “probability": {
                        "tumor": 0.75,
                        "stromal": 0.12,
                        "lymphocytes": 0.3,
                    },
                    "final_class": "tumor",
                    "embedding": NP array
                }
            }

            ATTENTION:
            - 'final_class' defines the class (tissue or cell) of the object.
        
        TIPS:
        - You can use h5py to read the h5 file.
        - You can use numpy to process the data.
        - You can use torch to process the data.
        - You can use some common libraries to process the data.

    VALID TISSUE TYPES (examples):
    [
    "tumor",
    "necrotic_tissue",
    "normal_tissue",
    "epithelial_tissue",
    "connective_tissue",
    "muscle_tissue",
    "nervous_tissue",
    "vascular_tissue",
    "adipose_tissue",
    "lymphatic_tissue",
    "bone_tissue",
    "cartilage_tissue",
    "fibrous_tissue",
    "glandular_tissue",
    "scar_tissue",
    "granulation_tissue",
    "mesenchymal_tissue",
    "endothelial_tissue"
    ]

    VALID CELL TYPES (examples):
    [
    "podocyte",
    "hepatocyte",
    "fibroblast",
    "endothelial_cell",
    "epithelial_cell",
    "keratinocyte",
    "melanocyte",
    "chondrocyte",
    "osteoblast",
    "osteoclast",
    "osteocyte",
    "myocyte",
    "cardiomyocyte",
    "smooth_muscle_cell",
    "skeletal_muscle_fiber",
    "neuron",
    "astrocyte",
    "oligodendrocyte",
    "microglia",
    "Schwann_cell",
    "macrophage",
    "monocyte",
    "neutrophil",
    "basophil",
    "eosinophil",
    "lymphocyte",
    "B_cell",
    "T_cell",
    "NK_cell",
    "erythrocyte",
    "megakaryocyte",
    "platelet",
    "beta_cell_of_pancreas",
    "alpha_cell_of_pancreas",
    "kupffer_cell"
    ]
    
    Sometimes, the function below will be used to give you the h5 file structure.
    def read_h5_structure(path, max_array_preview=5):
        def explore_group(group, indent=0):
            structure = []
            for key, item in group.items():
                prefix = "    " * indent

                if isinstance(item, h5py.Group):
                    structure.append(f"{prefix}{key}/ (Group)")
                    structure.extend(explore_group(item, indent + 1))

                elif isinstance(item, h5py.Dataset):
                    shape_str = f"shape={item.shape}" if len(
                        item.shape) > 0 else "scalar"
                    dtype_str = f"dtype={item.dtype}"

                    # Add data preview for small datasets or the first few elements
                    preview = ""
                    try:
                        data = item[()]
                        # Convert bytes to string if necessary
                        if isinstance(data, bytes):
                            data = data.decode('utf-8')

                        if isinstance(data, str):
                            preview = f"\n{prefix}    Preview: {data[:100]}..."
                            # Try to parse as JSON if it looks like a JSON string
                            if data.strip().startswith('{'):
                                try:
                                    dict_data = json.loads(data)
                                    structure.append(
                                        f"{prefix}    Dictionary content:")
                                    for k, v in dict_data.items():
                                        structure.append(
                                            f"{prefix}        {k}: {str(v)[:50]}..."
                                        )
                                except json.JSONDecodeError:
                                    pass
                        elif len(item.shape) == 0:
                            preview = f"\n{prefix}    Value: {data}"
                        elif np.prod(item.shape) > 0:
                            flat_data = data.flatten()
                            preview_data = flat_data[:max_array_preview]
                            preview = f"\n{prefix}    Preview: {preview_data}..."
                    except Exception as e:
                        preview = f"\n{prefix}    Preview unavailable: {str(e)}"

                    structure.append(
                        f"{prefix}{key}: ({shape_str}, {dtype_str}){preview}")

            return structure

        try:
            with h5py.File(path, 'r') as f:
                structure = ["H5 File Structure:"]
                structure.extend(explore_group(f))

            return '\n'.join(structure)

    except Exception as e:
        return f"Error reading H5 file: {str(e)}"


    
        EXAMPLE CODE:
        def analyze_medical_image(path):
            results = {}
            # Read the h5 file
            with h5py.File(path, 'r') as f:
                # Process the data
                # ...
            # Return the results
            return results

        Sometimes, there are message added after the user query from other agents to add additional information to the code.
        """

    response = script_agent.chat.completions.create(model="gpt-4o",
                                                    messages=[{
                                                        "role":
                                                        "system",
                                                        "content":
                                                        system_prompt
                                                    }, {
                                                        "role":
                                                        "user",
                                                        "content":
                                                        user_query
                                                    }])
    print(response)
    return response.choices[0].message.content

In [27]:
def read_h5_structure(path, max_array_preview=5):

    def explore_group(group, indent=0):
        structure = []
        for key, item in group.items():
            prefix = "    " * indent

            if isinstance(item, h5py.Group):
                structure.append(f"{prefix}{key}/ (Group)")
                structure.extend(explore_group(item, indent + 1))

            elif isinstance(item, h5py.Dataset):
                shape_str = f"shape={item.shape}" if len(
                    item.shape) > 0 else "scalar"
                dtype_str = f"dtype={item.dtype}"

                # Add data preview for small datasets or the first few elements
                preview = ""
                try:
                    data = item[()]
                    # Convert bytes to string if necessary
                    if isinstance(data, bytes):
                        data = data.decode('utf-8')

                    if isinstance(data, str):
                        preview = f"\n{prefix}    Preview: {data[:100]}..."
                        # Try to parse as JSON if it looks like a JSON string
                        if data.strip().startswith('{'):
                            try:
                                dict_data = json.loads(data)
                                structure.append(
                                    f"{prefix}    Dictionary content:")
                                for k, v in dict_data.items():
                                    structure.append(
                                        f"{prefix}        {k}: {str(v)[:50]}..."
                                    )
                            except json.JSONDecodeError:
                                pass
                    elif len(item.shape) == 0:
                        preview = f"\n{prefix}    Value: {data}"
                    elif np.prod(item.shape) > 0:
                        flat_data = data.flatten()
                        preview_data = flat_data[:max_array_preview]
                        preview = f"\n{prefix}    Preview: {preview_data}..."
                except Exception as e:
                    preview = f"\n{prefix}    Preview unavailable: {str(e)}"

                structure.append(
                    f"{prefix}{key}: ({shape_str}, {dtype_str}){preview}")

        return structure

    try:
        with h5py.File(path, 'r') as f:
            structure = ["H5 File Structure:"]
            structure.extend(explore_group(f))

        return '\n'.join(structure)

    except Exception as e:
        return f"Error reading H5 file: {str(e)}"


# Example usage:
# print(read_h5_structure('CMU-1.svs.h5'))
tip_from_agent = """
    MESSAGE FROM AGENT:
    If there are ceils, sample 5 random cells and return their contours. Also, put a color label for each cell. A cell is labeled red if it's contour is longer than 10 pixels; otherwise, it's labeled blue.
    This output should be in this format (example):
    
        "sample_cells": [
            {"color": "red", "contour": [x1, y1, x2, y2, ...]},
            {"color": "blue", "contour": [x1, y1, x2, y2, ...]},
            ...
        ]
    
    """
script = test_script(
    'Count the number of cells.\n'
    + read_h5_structure('CMU-1.svs.h5') + tip_from_agent)
print(script)

In [28]:
def script_function(script):
    namespace = {}
    try:
        # Execute the code in the isolated namespace
        exec(script, namespace)
        # Return the function object
        return namespace['analyze_medical_image']
    except Exception as e:
        print(f"Error executing code: {e}")
        return None

func =script_function(script)

In [29]:
print(func('CMU-1.svs.h5'))